In [28]:
import torch
import snntorch as snn
import matplotlib.pyplot as plt


### Shared helper setup

In [ ]:
from pathlib import Path
import sys

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "lif_utils.py").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import lif_utils as lu


### Configuration

In [ ]:
BETA = 0.9
STEPS = 30
INPUT_START = 5
INPUT_DURATION = 4
INPUT_STRENGTH = 0.30
DECAY = 0.8

INPUT_STRENGTHS = torch.linspace(0.10, 0.30, steps=21)
BUFFER_CONFIGS = [
    ("No buffer", None),
    ("2-frame buffer", 2),
    ("3-frame buffer", 3),
    ("4-frame buffer", 4),
    ("Decay buffer", "decay4"),
]


### Time-varying input without a buffer

In [ ]:
raw_inputs = lu.make_raw_inputs(
    INPUT_STRENGTH,
    steps=STEPS,
    input_start=INPUT_START,
    input_duration=INPUT_DURATION,
)

spk_raw, mem_raw = lu.run_lif_layer(
    raw_inputs.reshape(-1, 1),
    beta=BETA,
)
spk_raw = spk_raw.squeeze(1)
mem_raw = mem_raw.squeeze(1)

time = torch.arange(STEPS)

plt.figure(figsize=(10, 4))
plt.plot(time, raw_inputs, marker="o", label="input")
plt.plot(time, mem_raw, marker="o", label="membrane")
plt.step(time, spk_raw, where="post", label="spike")
plt.title("LIF response without buffer")
plt.xlabel("time step")
plt.legend()
plt.show()


### Compare buffer traces

In [ ]:
example_traces = {}

for label, buffer_type in BUFFER_CONFIGS:
    lif_inputs = lu.apply_buffer(
        raw_inputs,
        buffer_type,
        decay=DECAY,
    )
    spk_rec, mem_rec = lu.run_lif_layer(
        lif_inputs.reshape(-1, 1),
        beta=BETA,
    )
    example_traces[label] = {
        "input": lif_inputs,
        "spikes": spk_rec.squeeze(1),
        "membrane": mem_rec.squeeze(1),
    }

fig, axes = plt.subplots(
    len(example_traces),
    1,
    figsize=(10, 2.5 * len(example_traces)),
    sharex=True,
)

for ax, (label, trace) in zip(axes, example_traces.items()):
    ax.plot(time, trace["input"], marker="o", label="buffered input")
    ax.plot(time, trace["membrane"], marker="o", label="membrane")
    ax.step(time, trace["spikes"], where="post", label="spike")
    ax.set_title(label)
    ax.set_ylim(bottom=0)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("time step")
plt.tight_layout()
plt.show()


### Binary spike response sweep

In [ ]:
spike_sweep = {}
threshold_results = {}

fig, axes = plt.subplots(
    len(BUFFER_CONFIGS),
    1,
    figsize=(8, 2.2 * len(BUFFER_CONFIGS)),
    sharex=True,
)

for ax, (label, buffer_type) in zip(axes, BUFFER_CONFIGS):
    spikes = [
        lu.did_spike(
            float(input_strength),
            buffer_type,
            beta=BETA,
            steps=STEPS,
            input_start=INPUT_START,
            input_duration=INPUT_DURATION,
            decay=DECAY,
        )
        for input_strength in INPUT_STRENGTHS
    ]
    spike_sweep[label] = spikes
    threshold_results[label] = lu.first_spike_threshold(
        INPUT_STRENGTHS,
        buffer_type,
        beta=BETA,
        steps=STEPS,
        input_start=INPUT_START,
        input_duration=INPUT_DURATION,
        decay=DECAY,
    )

    ax.plot(
        INPUT_STRENGTHS.numpy(),
        spikes,
        marker="o",
        drawstyle="steps-post",
    )
    ax.set_title(label)
    ax.set_ylabel("spike")
    ax.set_yticks([0, 1])

axes[-1].set_xlabel("input strength")
plt.tight_layout()
plt.show()

threshold_results


### Threshold summary

In [ ]:
for label, threshold in threshold_results.items():
    if threshold is None:
        print(f"{label}: no spike")
    else:
        print(f"{label}: {threshold:.2f}")


In [ ]:
labels = list(threshold_results.keys())
thresholds = [threshold_results[label] for label in labels]

plt.figure(figsize=(8, 4))
plt.bar(
    labels,
    [float("nan") if threshold is None else threshold for threshold in thresholds],
)
plt.ylabel("first input strength that spikes")
plt.title("Spike threshold by buffer type")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
